# Datenbasis: Quelle, Verknüpfung und Auswirkungen

Dieses Notebook rekonstruiert die **ungekappten** Analysetabellen aus dem festgelegten CASdatasets-Datenstand. Es verwendet ausschließlich deterministische Quellprüfungen und Zielabgleiche. Es schätzt kein Modell und wählt keine Merkmale oder Ausschlüsse anhand von Ergebnissen. Die explorative Analyse und die Modellbewertung folgen erst nach Festlegung der Validierungsgrenzen.

Vor dem ersten Lauf im Projektstamm `python -m src.data --download` ausführen. Dieses Notebook lädt keine Dateien aus dem Netz nach.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "config/data_contract.json").is_file()
             and (p / "src/data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Notebook im Projektstamm oder einem Unterverzeichnis starten.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from src.data import prepare, load_prepared

report, directory = prepare()  # nur geprüfter lokaler Rohdaten-Cache
policies, claims, verified = load_prepared(directory)
assert report == verified
print("Quellenkennung:", report["source_id"])
print("Vorbereitungskennung:", report["preparation_id"])

## Zielgröße und Einheiten

Die Frequenzzielgröße zählt **erfasste Schadenzeilen mit positivem Betrag je Vertragsperiode**. `ClaimNb` bleibt als mitgelieferte Anzahl erhalten; `recorded_positive_count` wird unabhängig aus den Schadenzeilen rekonstruiert. `recorded_loss_eur` umfasst deren Summe je Vertrag, einschließlich Null bei schadenfreien Verträgen. `Exposure` wird in Vertragsjahren gemessen. Ein Jahreswert und ein Wert der tatsächlichen Vertragsperiode sind unterschiedliche Größen.

Die Datei enthält keine echte Schaden-ID. `claim_row_id` kennzeichnet nur Rohdatei und Zeilenposition. Gleiche Vertragsnummer und gleicher Betrag belegen keine Dublette.

In [ ]:
display(pd.DataFrame(report["contract"]["units"].items(),
                     columns=["Spalte", "Bedeutung / Einheit"]))
print("Quelle:", report["source"]["version"], report["source"]["commit"])

## Erhaltene Daten und Verknüpfung

Verträge ohne Schaden bleiben Teil des Portfolios. Schadenzeilen ohne passenden Vertrag und Unterschiede zwischen mitgelieferter und rekonstruierter Anzahl würden den Aufbau abbrechen. Alle Ausschlüsse müssten begründet und bilanziert werden; die primäre Datenbasis nimmt keine vor.

In [ ]:
labels = {"policies": "Vertragszeilen", "exposure_years": "Exponierung [Vertragsjahre]",
          "supplied_claims": "Mitgelieferte Anzahl",
          "recorded_positive_claims": "Rekonstruierte positive Schadenzeilen",
          "recorded_loss_eur": "Erfasster Schadenbetrag [EUR]"}
reconciliation = report["reconciliation"]
display(pd.DataFrame({"Größe": [labels[k] for k in labels],
                      "Erhalten": [reconciliation["retained"][k] for k in labels]}))
display(pd.DataFrame([
    {"Prüfung": "Verträge ohne Schadenzeilen", **reconciliation["anti_joins"]["policy_without_claim_rows"]},
    {"Prüfung": "Verträge mit unterschiedlicher Anzahl", **reconciliation["supplied_vs_derived_mismatch"]}
]).rename(columns=labels))
display(pd.DataFrame([reconciliation["anti_joins"]["claim_rows_without_policy"]])
        .rename(columns={"claim_rows": "Schadenzeilen ohne Vertrag", "policies": "Quellschlüssel",
                         "recorded_loss_eur": "Nicht verknüpfter Betrag [EUR]"}))
print("Unverändert erhaltene exakt gleiche Schadenzeilen über die erste Zeile hinaus:",
      reconciliation["exact_repeat_claim_rows_beyond_first_retained"])
display(pd.DataFrame(reconciliation["count_cross_table"])
        .rename(columns={"supplied": "Mitgelieferte Anzahl", "derived": "Rekonstruierte Anzahl",
                         "policies": "Vertragszeilen"}))

## Prüfgruppen und Extremwerte

Die folgenden Gruppen überlappen sich. Sie dokumentieren Umfang und Schadenunterstützung; sie begründen weder eine automatische Korrektur noch einen kausalen Zusammenhang. Die Entstehung von Exponierungen über einem Jahr und Alterswerten von 100 ist mit den verfügbaren Informationen nicht geklärt. Markenklassen sind anonymisiert. Insbesondere ist B12 keine bekannte Marken- oder Fahrzeugklasse.

In [ ]:
audit = report["impact_audit"]
flag_labels = {"exposure_above_one": "Exponierung > 1 Jahr",
               "exposure_below_one_month": "Exponierung < 1/12 Jahr",
               "driver_age_above_90": "Fahreralter > 90",
               "vehicle_age_above_40": "Fahrzeugalter > 40",
               "driver_age_100": "Fahreralter = 100",
               "vehicle_age_100": "Fahrzeugalter = 100",
               "count_above_four": "Schadenanzahl > 4",
               "bonus_malus_above_150": "Bonus-Malus > 150",
               "brand_B12": "Markenklasse B12",
               "brand_B12_new_vehicle": "B12 und Fahrzeugalter = 0"}
display(pd.DataFrame([{ "Prüfgruppe": flag_labels[name], **values}
                      for name, values in audit["policy_flags"].items()]).rename(columns=labels))
display(pd.DataFrame(audit["ranges"]).T.rename(columns={"min": "Minimum", "max": "Maximum"}))
display(pd.DataFrame(audit["claim_amount_quantiles_eur"].items(),
                     columns=["Quantil", "Schadenbetrag [EUR]"]))

## Was hypothetische Kappungen verändern würden

Die Analysetabellen werden **nicht** gekappt. Eine Exponierungskappung verändert den Nenner der Jahresrate. Eine Anzahlkappung verändert die Zielgröße und unterbricht den Gleichlauf zu den unveränderten Schadenzeilen. Für entfernte Anzahlen wird daher kein willkürlicher Eurobetrag zugeordnet. Eine Schadenbetragskappung verändert den erwarteten Schadenaufwand. Die folgenden Beträge quantifizieren diese Eingriffe; sie sind keine Empfehlung für einen Schwellenwert.

In [ ]:
display(pd.DataFrame([audit["hypothetical_exposure_cap_one"]]).rename(columns={
    "removed_policy_years": "Bei Kappung auf 1 entfernte Vertragsjahre",
    "remaining_policy_years": "Verbleibende Vertragsjahre",
    "unchanged_claims": "Unveränderte Schadenzeilen", "unchanged_loss_eur": "Unveränderter Betrag [EUR]"}))
display(pd.DataFrame([audit["hypothetical_count_cap_four"]]).rename(columns={
    "removed_counts": "Bei Kappung auf 4 entfernte Anzahlen",
    "remaining_counts": "Verbleibende Anzahlen", "loss_allocation": "Eurozuordnung"}))
display(pd.DataFrame([{ "Kappung je Schaden [EUR]": cap, **values}
                      for cap, values in audit["hypothetical_claim_amount_caps_eur"].items()])
        .rename(columns={"claim_rows": "Betroffene Schadenzeilen", "policies": "Betroffene Verträge",
                         "recorded_loss_eur": "Betrag der betroffenen Schäden [EUR]",
                         "removed_loss_eur": "Entfernter Betrag [EUR]",
                         "remaining_loss_eur": "Verbleibender Gesamtbetrag [EUR]"}))

## Nachweis und nächste Analysegrenze

Der Download prüft festgelegte Rohdatei-Hashes. Der Vorbereitungsnachweis enthält Quellstand, Einheiten, Schema, Vertrags-/Schadenabgleich, Konfiguration, Code- und Paketversionen sowie die Hashes beider Analysetabellen. `load_prepared` prüft die Tabellen vor der Verwendung.

Die bisherigen Ausgaben sind Datenbuchhaltung über den vollständigen Datenstand. Eine spätere explorative Untersuchung oder Modellwahl benötigt festgelegte Entwicklungs- und Evaluationsgrenzen auf Vertragsebene. Es liegen hier keine OOF-Vorhersagen, adjustierten Alterseffekte oder neuen Modellgütewerte vor.

In [ ]:
display(pd.DataFrame([{ "Tabelle": name, "Zeilen": values["rows"],
                        "Datei": values["filename"], "SHA-256": values["sha256"]}
                       for name, values in report["analysis_tables"].items()]))
assert policies["IDpol"].is_unique
assert len(claims) == int(policies["recorded_positive_count"].sum())
assert policies["ClaimNb"].equals(policies["recorded_positive_count"])
print("Deterministische Vorbereitung und Tabellenprüfung abgeschlossen.")